# 第 4 章互动实验：透视扭曲与矫正

本实验生成网格并调用 OpenCV 作为几何基线，不包含 `4-1_齐次点变换.py` 的答案。拖动透视强度，观察有效像素、插值误差和点顺序。

In [ ]:
%matplotlib inline
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider

height, width = 180, 240
grid = np.full((height, width), 245, dtype=np.uint8)
for x in range(0, width, 20): cv2.line(grid, (x, 0), (x, height-1), 80, 1)
for y in range(0, height, 20): cv2.line(grid, (0, y), (width-1, y), 80, 1)
cv2.putText(grid, 'TOP', (85, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.7, 0, 2)
src = np.float32([[0, 0], [width-1, 0], [width-1, height-1], [0, height-1]])

In [ ]:
def show_perspective(tilt=35):
    dst = np.float32([[tilt, 8], [width-1-tilt//2, 0], [width-1, height-1], [0, height-1-tilt//3]])
    H = cv2.getPerspectiveTransform(src, dst)
    warped = cv2.warpPerspective(grid, H, (width, height), flags=cv2.INTER_LINEAR, borderValue=255)
    H_inv = cv2.getPerspectiveTransform(dst, src)
    restored = cv2.warpPerspective(warped, H_inv, (width, height), flags=cv2.INTER_LINEAR, borderValue=255)
    mae = float(np.mean(np.abs(restored.astype(np.float32) - grid.astype(np.float32))))
    projected = cv2.perspectiveTransform(src.reshape(-1, 1, 2), H).reshape(-1, 2)
    corner_error = float(np.max(np.linalg.norm(projected - dst, axis=1)))
    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    for ax, image, title in zip(axes, [grid, warped, restored], ['原网格', '透视扭曲', f'反变换：MAE={mae:.2f}']):
        ax.imshow(image, cmap='gray', vmin=0, vmax=255)
        ax.set_title(title); ax.axis('off')
    plt.suptitle(f'四角重投影最大误差 {corner_error:.2e} 像素')
    plt.tight_layout(); plt.show()

interact(show_perspective, tilt=IntSlider(min=0, max=90, step=5, value=35));

## 破坏实验与记录

1. 把 `tilt` 提高，观察反变换虽能运行但插值误差如何上升。
2. 在副本中交换两个 `dst` 点，观察图像翻折或自交。
3. 思考为什么四角重投影误差很小，恢复图像仍不等于原图。

把矩阵、点顺序、MAE 和失败原因写入 `实验记录.md`。